# 1. Prepare data

In [2]:
import numpy as np
import pandas as pd
import ast
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# ─────────────────────────────────────────────
# 1. HELPER FUNCTIONS
# ─────────────────────────────────────────────

def parse_complex_array(s: str) -> np.ndarray:
    """Parse a string representation of a complex numpy array into np.ndarray."""
    s = s.strip()
    # Remove surrounding brackets
    if s.startswith("["):
        s = s[1:]
    if s.endswith("]"):
        s = s[:-1]
    # Split on comma
    tokens = [t.strip() for t in s.split(",") if t.strip()]
    complexes = []
    for t in tokens:
        # Handle "real imag j" with a space between real and signed imaginary,
        # e.g. "2.6537056 -6.91795864e-04j"  →  join without space before parsing
        t_clean = t.replace(" ", "")
        # After removing spaces a leading sign may be lost if both parts were
        # separated: "2.65 -0.03j" → "2.65-0.03j"  (still valid for complex())
        complexes.append(complex(t_clean))
    return np.array(complexes)


def extract_features(z_array: np.ndarray) -> np.ndarray:
    """
    Extract statistical features from a complex impedance spectrum.

    For each of the 4 components (real, imaginary, magnitude, phase),
    compute: mean, std, min, max  →  4 × 4 = 16 features per sample.
    """
    real_part = z_array.real
    imag_part = z_array.imag
    magnitude  = np.abs(z_array)
    phase      = np.angle(z_array)          # radians

    features = []
    for component in [real_part, imag_part, magnitude, phase]:
        features.extend([
            np.mean(component),
            np.std(component),
            np.min(component),
            np.max(component),
        ])
    return np.array(features)


def load_and_extract(filepath: str):
    """Load a CSV file and return (X, y) after feature extraction."""
    df = pd.read_csv(filepath)

    X_list, y_list = [], []
    for _, row in df.iterrows():
        z_array = parse_complex_array(row["Z"])
        feats   = extract_features(z_array)
        X_list.append(feats)
        y_list.append(row["Circuit"])

    X = np.vstack(X_list)
    y = np.array(y_list)
    return X, y


# ─────────────────────────────────────────────
# 2. LOAD DATA
# ─────────────────────────────────────────────

print("Loading train_data.csv ...")
X_train, y_train = load_and_extract("dataset/train_data.csv")

print("Loading test_data.csv ...")
X_test_full, y_test_full = load_and_extract("dataset/test_data.csv")

# ─────────────────────────────────────────────
# 3. SPLIT test_data → validation (20%) + test (80%)
# ─────────────────────────────────────────────

X_val, X_test, y_val, y_test = train_test_split(
    X_test_full, y_test_full,
    test_size=0.80,
    random_state=42,
    stratify=y_test_full,
)

print(f"\nDataset sizes:")
print(f"  Train      : {X_train.shape}  —  {len(X_train)} samples")
print(f"  Validation : {X_val.shape}  —  {len(X_val)} samples  (20 % of test_data.csv)")
print(f"  Test       : {X_test.shape}  —  {len(X_test)} samples  (80 % of test_data.csv)")
print(f"\nFeature vector: {X_train.shape[1]} features")
print(f"  real  : mean, std, min, max")
print(f"  imag  : mean, std, min, max")
print(f"  |Z|   : mean, std, min, max")
print(f"  phase : mean, std, min, max")

# ─────────────────────────────────────────────
# 4. ENCODE LABELS
# ─────────────────────────────────────────────

le = LabelEncoder()
le.fit(y_train)                     # fit on training labels only

y_train_enc = le.transform(y_train)
y_val_enc   = le.transform(y_val)
y_test_enc  = le.transform(y_test)

print(f"\nClasses ({len(le.classes_)}): {list(le.classes_)}")

# ─────────────────────────────────────────────
# 5. SCALE FEATURES
# ─────────────────────────────────────────────

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)      # fit only on train
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# ─────────────────────────────────────────────
# 6. TRAIN SVM
# ─────────────────────────────────────────────

print("\nTraining SVM ...")
svm = SVC(kernel="rbf", C=10, gamma="scale", random_state=42)
svm.fit(X_train_scaled, y_train_enc)

# ─────────────────────────────────────────────
# 7. EVALUATE
# ─────────────────────────────────────────────

print("\n── Validation Results ──────────────────────────────")
y_val_pred = svm.predict(X_val_scaled)
print(classification_report(
    y_val_enc, y_val_pred,
    target_names=le.classes_,
    zero_division=0,
))

print("── Test Results ────────────────────────────────────")
y_test_pred = svm.predict(X_test_scaled)
print(classification_report(
    y_test_enc, y_test_pred,
    target_names=le.classes_,
    zero_division=0,
))

Loading train_data.csv ...
Loading test_data.csv ...

Dataset sizes:
  Train      : (7462, 16)  —  7462 samples
  Validation : (373, 16)  —  373 samples  (20 % of test_data.csv)
  Test       : (1492, 16)  —  1492 samples  (80 % of test_data.csv)

Feature vector: 16 features
  real  : mean, std, min, max
  imag  : mean, std, min, max
  |Z|   : mean, std, min, max
  phase : mean, std, min, max

Classes (9): ['L-R-RCPE', 'L-R-RCPE-RCPE', 'L-R-RCPE-RCPE-RCPE', 'RC-G-G', 'RC-RC-RCPE-RCPE', 'RCPE-RCPE', 'RCPE-RCPE-RCPE', 'RCPE-RCPE-RCPE-RCPE', 'Rs_Ws']

Training SVM ...

── Validation Results ──────────────────────────────
                     precision    recall  f1-score   support

           L-R-RCPE       0.33      0.52      0.40        44
      L-R-RCPE-RCPE       0.16      0.10      0.12        41
 L-R-RCPE-RCPE-RCPE       0.53      0.17      0.26        46
             RC-G-G       0.66      0.50      0.57        46
    RC-RC-RCPE-RCPE       0.30      0.74      0.43        46
        